In [2]:
import wandb
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# API key'ini buraya yapıştır
wandb.login(key="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")

print("Giriş başarılı!")
print("Wandb versiyonu:", wandb.__version__)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\PC\_netrc
wandb: Currently logged in as: safkyigt (safkyigt-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Giriş başarılı!
Wandb versiyonu: 0.25.1


In [3]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# Veri
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

# Model
class CNN(nn.Module):
    def __init__(self, dropout_rate):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3136, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x))

print("Model ve veri hazır!")

Model ve veri hazır!


In [1]:
def train_with_wandb(config):
    # W&B run başlat
    run = wandb.init(
        project="mnist-cnn",
        config=config,
        name=f"lr={config['lr']}_dropout={config['dropout']}"
    )

    model = CNN(dropout_rate=config['dropout'])
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])

    for epoch in range(config['epochs']):
        # Eğitim
        model.train()
        total_loss, correct = 0, 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (output.argmax(1) == y_batch).sum().item()

        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_dataset)

        # Test
        model.eval()
        total_loss, correct = 0, 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                output = model(X_batch)
                loss = criterion(output, y_batch)
                total_loss += loss.item()
                correct += (output.argmax(1) == y_batch).sum().item()

        test_loss = total_loss / len(test_loader)
        test_acc = correct / len(test_dataset)

        # W&B'ye logla — işte sihir burada!
        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_loss,
            "test_acc": test_acc
        })

        print(f"Epoch {epoch+1}/{config['epochs']} | Train Acc: {train_acc:.2%} | Test Acc: {test_acc:.2%}")

    wandb.finish()
    return test_acc

# İlk deneyi çalıştır
config = {
    "lr": 0.001,
    "dropout": 0.3,
    "epochs": 3,
    "batch_size": 64
}

print("Deney başlıyor...")
train_with_wandb(config)

Deney başlıyor...


NameError: name 'wandb' is not defined

In [2]:
import wandb
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# Giriş
wandb.login()

# Veri
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

# Model
class CNN(nn.Module):
    def __init__(self, dropout_rate):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3136, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x))

def train_with_wandb(config):
    run = wandb.init(
        project="mnist-cnn",
        config=config,
        name=f"lr={config['lr']}_dropout={config['dropout']}"
    )

    model = CNN(dropout_rate=config['dropout'])
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])

    for epoch in range(config['epochs']):
        model.train()
        total_loss, correct = 0, 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (output.argmax(1) == y_batch).sum().item()

        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_dataset)

        model.eval()
        total_loss, correct = 0, 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                output = model(X_batch)
                loss = criterion(output, y_batch)
                total_loss += loss.item()
                correct += (output.argmax(1) == y_batch).sum().item()

        test_loss = total_loss / len(test_loader)
        test_acc = correct / len(test_dataset)

        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_loss,
            "test_acc": test_acc
        })

        print(f"Epoch {epoch+1}/{config['epochs']} | Train Acc: {train_acc:.2%} | Test Acc: {test_acc:.2%}")

    wandb.finish()

# Deneyi çalıştır
config = {
    "lr": 0.001,
    "dropout": 0.3,
    "epochs": 3,
    "batch_size": 64
}

print("Deney başlıyor...")
train_with_wandb(config)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\PC\_netrc.
wandb: Currently logged in as: safkyigt (safkyigt-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Deney başlıyor...


ServicePollForTokenError: Failed to read port info after 30.0 seconds.

In [1]:
import os
os.environ["WANDB_MODE"] = "offline"

import wandb
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

wandb.login()

# Veri
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

class CNN(nn.Module):
    def __init__(self, dropout_rate):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3136, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x))

def train_with_wandb(config):
    run = wandb.init(
        project="mnist-cnn",
        config=config,
        name=f"lr={config['lr']}_dropout={config['dropout']}"
    )

    model = CNN(dropout_rate=config['dropout'])
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])

    for epoch in range(config['epochs']):
        model.train()
        total_loss, correct = 0, 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (output.argmax(1) == y_batch).sum().item()

        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_dataset)

        model.eval()
        total_loss, correct = 0, 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                output = model(X_batch)
                loss = criterion(output, y_batch)
                total_loss += loss.item()
                correct += (output.argmax(1) == y_batch).sum().item()

        test_loss = total_loss / len(test_loader)
        test_acc = correct / len(test_dataset)

        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_loss,
            "test_acc": test_acc
        })

        print(f"Epoch {epoch+1}/{config['epochs']} | Train Acc: {train_acc:.2%} | Test Acc: {test_acc:.2%}")

    wandb.finish()
    print(f"\nRun klasörü: {run.dir}")

config = {
    "lr": 0.001,
    "dropout": 0.3,
    "epochs": 3,
    "batch_size": 64
}

print("Deney başlıyor...")
train_with_wandb(config)

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


Deney başlıyor...
Epoch 1/3 | Train Acc: 93.69% | Test Acc: 98.49%
Epoch 2/3 | Train Acc: 97.95% | Test Acc: 98.82%
Epoch 3/3 | Train Acc: 98.52% | Test Acc: 98.97%


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core



Run klasörü: c:\Users\PC\OneDrive\Masaüstü\deep-learning\wandb\offline-run-20260314_210753-aaa2vuql\files


In [2]:
import os
os.environ["WANDB_MODE"] = "offline"
import wandb
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# Veri
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

class CNN(nn.Module):
    def __init__(self, dropout_rate):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(), nn.Linear(3136, 128), nn.ReLU(),
            nn.Dropout(dropout_rate), nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.fc_layers(self.conv_layers(x))

# Sweep konfigürasyonu
sweep_config = {
    "method": "grid",
    "metric": {"name": "test_acc", "goal": "maximize"},
    "parameters": {
        "lr":      {"values": [0.001, 0.0001]},
        "dropout": {"values": [0.2, 0.4]},
    }
}

def sweep_train():
    run = wandb.init()
    config = wandb.config

    model = CNN(dropout_rate=config.dropout)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

    for epoch in range(3):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                correct += (model(X_batch).argmax(1) == y_batch).sum().item()
        test_acc = correct / len(test_dataset)
        wandb.log({"test_acc": test_acc, "epoch": epoch})

    print(f"lr={config.lr} dropout={config.dropout} → Test Acc: {test_acc:.2%}")
    wandb.finish()

# Sweep başlat — 4 farklı kombinasyon dener
sweep_id = wandb.sweep(sweep_config, project="mnist-sweep")
wandb.agent(sweep_id, sweep_train, count=4)

Create sweep with ID: d3j580q2
Sweep URL: https://wandb.ai/safkyigt-/mnist-sweep/sweeps/d3j580q2


wandb: Agent Starting Run: 7g4xwmlk with config:
wandb: 	dropout: 0.2
wandb: 	lr: 0.001
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


lr=0.001 dropout=0.2 → Test Acc: 99.06%


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: Agent Starting Run: xgu44z5o with config:
wandb: 	dropout: 0.2
wandb: 	lr: 0.0001
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


lr=0.0001 dropout=0.2 → Test Acc: 98.03%


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: Agent Starting Run: znxhr8ad with config:
wandb: 	dropout: 0.4
wandb: 	lr: 0.001
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


lr=0.001 dropout=0.4 → Test Acc: 98.82%


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: ppiv2h7v with config:
wandb: 	dropout: 0.4
wandb: 	lr: 0.0001
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


lr=0.0001 dropout=0.4 → Test Acc: 98.16%


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
